## News Category Classification Model
### Full Fine-tuning DistilBert model

In [1]:
# generate hugginface token or login to hf account
import huggingface_hub
huggingface_hub.login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [2]:
# import the necessary libraries and modules
try:
    import datasets, evaluate, accelerate
    import gradio as gr
except ModuleNotFoundError:
    !pip install datasets evaluate accelerate gradio
    import datasets, evaluate, accelerate
    import gradio as gr

import torch
import transformers
import random
import numpy as np
import pandas as pd

# see version of the libraries
print(f"transformers version: {transformers.__version__}")
print(f"datasets version: {datasets.__version__}")
print(f"evaluate version: {evaluate.__version__}")
print(f"accelerate version: {accelerate.__version__}")
print(f"gradio version: {gr.__version__}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00
transformers version: 5.15.1
datasets version: 4.0.0
evaluate version: 0.4.6
accelerate version: 1.14.0
gradio version: 6.26.0


### Prepare Dataset

In [3]:
from datasets import load_dataset

# load the dataset
dataset = load_dataset(path="AiresPucrs/News-Category-Dataset")
dataset

README.md:   0%|          | 0.00/847 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 27.5MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/209527 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 209527
    })
})

In [4]:
print(dataset.column_names)
print(dataset['train'].features)
print(dataset['train'][0])

{'train': ['text', 'labels']}
{'text': Value('string'), 'labels': Value('string')}
{'text': 'Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.', 'labels': 'U.S. NEWS'}


In [5]:
# unique labels in the dataset
labels = dataset['train'].unique("labels")
print(f"Unique labels: {labels}")
print(f"Number of unique labels: {len(labels)}")

Unique labels: ['U.S. NEWS', 'COMEDY', 'PARENTING', 'WORLD NEWS', 'CULTURE & ARTS', 'TECH', 'SPORTS', 'ENTERTAINMENT', 'POLITICS', 'WEIRD NEWS', 'ENVIRONMENT', 'EDUCATION', 'CRIME', 'SCIENCE', 'WELLNESS', 'BUSINESS', 'STYLE & BEAUTY', 'FOOD & DRINK', 'MEDIA', 'QUEER VOICES', 'HOME & LIVING', 'WOMEN', 'BLACK VOICES', 'TRAVEL', 'MONEY', 'RELIGION', 'LATINO VOICES', 'IMPACT', 'WEDDINGS', 'COLLEGE', 'PARENTS', 'ARTS & CULTURE', 'STYLE', 'GREEN', 'TASTE', 'HEALTHY LIVING', 'THE WORLDPOST', 'GOOD NEWS', 'WORLDPOST', 'FIFTY', 'ARTS', 'DIVORCE']
Number of unique labels: 42


In [6]:
# turn the dataset into a pandas dataframe and check some samples
news_df = pd.DataFrame(dataset['train'])
news_df.sample(5)

,text,labels
95584,The Well-Meaning Friend,HEALTHY LIVING
173968,Valentine's Day: PR's Favorite Holiday Since T...,FOOD & DRINK
6189,Far-Right Terror Attacks Possible 'In Coming Y...,WORLD NEWS
167536,A Dozen Menu Items McDonald's Should Import No...,FOOD & DRINK
10793,"Jordan Peele, BuzzFeed Create Fake News Video ...",ENTERTAINMENT


In [7]:
# create mapping of labels to numeric values
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for idx, label in enumerate(labels)}
print(f"Label to ID mapping: {label2id}")
print(f"ID to Label mapping: {id2label}")

Label to ID mapping: {'U.S. NEWS': 0, 'COMEDY': 1, 'PARENTING': 2, 'WORLD NEWS': 3, 'CULTURE & ARTS': 4, 'TECH': 5, 'SPORTS': 6, 'ENTERTAINMENT': 7, 'POLITICS': 8, 'WEIRD NEWS': 9, 'ENVIRONMENT': 10, 'EDUCATION': 11, 'CRIME': 12, 'SCIENCE': 13, 'WELLNESS': 14, 'BUSINESS': 15, 'STYLE & BEAUTY': 16, 'FOOD & DRINK': 17, 'MEDIA': 18, 'QUEER VOICES': 19, 'HOME & LIVING': 20, 'WOMEN': 21, 'BLACK VOICES': 22, 'TRAVEL': 23, 'MONEY': 24, 'RELIGION': 25, 'LATINO VOICES': 26, 'IMPACT': 27, 'WEDDINGS': 28, 'COLLEGE': 29, 'PARENTS': 30, 'ARTS & CULTURE': 31, 'STYLE': 32, 'GREEN': 33, 'TASTE': 34, 'HEALTHY LIVING': 35, 'THE WORLDPOST': 36, 'GOOD NEWS': 37, 'WORLDPOST': 38, 'FIFTY': 39, 'ARTS': 40, 'DIVORCE': 41}
ID to Label mapping: {0: 'U.S. NEWS', 1: 'COMEDY', 2: 'PARENTING', 3: 'WORLD NEWS', 4: 'CULTURE & ARTS', 5: 'TECH', 6: 'SPORTS', 7: 'ENTERTAINMENT', 8: 'POLITICS', 9: 'WEIRD NEWS', 10: 'ENVIRONMENT', 11: 'EDUCATION', 12: 'CRIME', 13: 'SCIENCE', 14: 'WELLNESS', 15: 'BUSINESS', 16: 'STYLE & BE

In [8]:
# Map the labels in the dataset to their corresponding numeric values
def map_labels(example):
    example['labels'] = label2id[example['labels']]
    return example

label_mapped_dataset = dataset.map(map_labels)
# check some sample data after mapping
print(label_mapped_dataset['train'][5:10])

Map:   0%|          | 0/209527 [00:00<?, ? examples/s]

{'text': ['Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police The 63-year-old woman was seen working at the South Carolina store on Thursday. She was found dead Monday after her family reported her missing, authorities said.', 'Reporter Gets Adorable Surprise From Her Boyfriend While Live On TV "Who\'s that behind you?" an anchor for New York’s PIX11 asked journalist Michelle Ross as she finished up an interview.', 'Puerto Ricans Desperate For Water After Hurricane Fiona’s Rampage More than half a million people remained without water service three days after the storm lashed the U.S. territory.', 'How A New Documentary Captures The Complexity Of Being A Child Of Immigrants In "Mija," director Isabel Castro combined music documentaries with the style of "Euphoria" and "Clueless" to tell a more nuanced immigration story.', "Biden At UN To Call Russian War An Affront To Body's Charter White House officials say the crux of the president's visit to the U.N. this year wi

In [9]:

label_mapped_dataset['train'].shuffle()[:5]


{'text': ['REPORT: Police Seize Hundreds Of Millions From People Not Charged With Crimes ',
  'Michael Caine Slams Young Actors Who Just Want To Be \'Rich And Famous\' "They do a little part on television and everyone knows who they are. They can\'t really act."',
  'Lady GaGa: Divorce Is Not An Option For Me Lady GaGa has revealed that when she finds her Mr Right, she will stick with him through thick and thin. Read more on www.mtv.co.uk',
  "What It Takes To Get The New York Giants To An Away Game The suite life? Depending on where they're going, Phelan said, the Giants reserve up to 200 rooms in a hotel, with enough",
  "Here's What Happened When We Created A Space Just For Parents And Kids To Talk Lots of smiles, and a few tears. #TalkToMe"],
 'labels': [8, 7, 41, 23, 30]}

In [10]:
from datasets import DatasetDict

# split the dataset into train, validation and test sets
train_test_val_dataset = label_mapped_dataset['train'].train_test_split(test_size=0.2, seed=42)
# This results in 10% validation and 10% test relative to the original data
test_valid = train_test_val_dataset["test"].train_test_split(test_size=0.5, seed=42)

# Pack everything into a unified DatasetDict
final_dataset = DatasetDict({
    "train": train_test_val_dataset["train"],
    "validation": test_valid["train"],  # The 'train' part of the second split
    "test": test_valid["test"]          # The 'test' part of the second split
})

print(final_dataset)
final_dataset['test'].shuffle()[:5]


DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 167621
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 20953
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 20953
    })
})


{'text': ['‘Bad Guys’ Repeats At No. 1, Liam Neeson’s Latest Misfires The DreamWorks animated heist movie\xa0“The Bad Guys”\xa0was the top film in U.S. and Canada theaters for the second straight weekend.',
  'Diffused Congruence Podcast: Professor Zareena Grewal on Islam is a Foreign Country ',
  'Senate Passes Budget Blueprint Key To Trump Tax Effort The measure would add up to $1.5 trillion to the federal deficit over the next decade to pay for proposed tax cuts.',
  'Suspect In Pennsylvania Shooting Rampage Found Dead (UPDATE) ',
  'Sports Illustrated Swimsuit Cover 2013: Kate Upton Again, According To Leaked Photo (UPDATED) ... Drum roll please ...'],
 'labels': [7, 25, 8, 12, 16]}

### Prepare Tokenizer

In [11]:
from transformers import AutoTokenizer

# load the tokenizer for the model we want to use
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert/distilbert-base-uncased", use_fast=True)

tokenizer

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [12]:
# test the tokenizer on a sample text
sample_text = "This is a sample text for tokenization."
tokenized_output = tokenizer(sample_text)
tokenized_output

{'input_ids': [101, 2023, 2003, 1037, 7099, 3793, 2005, 19204, 3989, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [13]:
# check the maximum sequence length of the tokenizer
max_seq_length = tokenizer.model_max_length
vocab_size = tokenizer.vocab_size
print(f"Maximum sequence length of the tokenizer: {max_seq_length}, vocabulary size: {vocab_size}")

Maximum sequence length of the tokenizer: 512, vocabulary size: 30522


In [14]:
# define a tokenization function to apply to the dataset texts
def tokenize_text(example):
    return tokenizer(example['text'], padding=True, truncation=True)

# test the function on a sample text
sample_example= {'text': "The official Google Colab extension for VS Code does not natively support the Secrets (Key icon) user-data feature. Because the extension runs the notebook inside the VS Code Jupyter interface rather than the standard web UI, the google.colab.userdata module will fail to fetch keys stored in your browser-based Colab secrets panel", 'labels': 5}

tokenized_text_sample = tokenize_text(sample_example)
tokenized_text_sample

{'input_ids': [101, 1996, 2880, 8224, 15270, 2497, 5331, 2005, 5443, 3642, 2515, 2025, 3128, 2135, 2490, 1996, 7800, 1006, 3145, 12696, 1007, 5310, 1011, 2951, 3444, 1012, 2138, 1996, 5331, 3216, 1996, 14960, 2503, 1996, 5443, 3642, 18414, 7685, 3334, 8278, 2738, 2084, 1996, 3115, 4773, 21318, 1010, 1996, 8224, 1012, 15270, 2497, 1012, 5310, 2850, 2696, 11336, 2097, 8246, 2000, 18584, 6309, 8250, 1999, 2115, 16602, 1011, 2241, 15270, 2497, 7800, 5997, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [15]:
final_dataset
# Map the tokenization function to the entire dataset
tokenized_dataset = final_dataset.map(function=tokenize_text, batched=True, batch_size=1000)
tokenized_dataset

Map:   0%|          | 0/167621 [00:00<?, ? examples/s]

Map:   0%|          | 0/20953 [00:00<?, ? examples/s]

Map:   0%|          | 0/20953 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 167621
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20953
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20953
    })
})

In [16]:
# see two samples from the tokenized dataset with their all keys
tokenized_dataset['train'].shuffle()[:2]

{'text': ['Colbert Mocks Trump For Doing Exactly What \'The Crooked Clintons\' Did "Late Show" host tears into Trump over latest White House hire.',
  'How One Newspaper Improved Its Coverage Of An Underserved Community The Peoria, Illinois paper is doing so without hiring more reporters.'],
 'labels': [1, 18],
 'input_ids': [[101,
   23928,
   12934,
   2015,
   8398,
   2005,
   2725,
   3599,
   2054,
   1005,
   1996,
   15274,
   7207,
   2015,
   1005,
   2106,
   1000,
   2397,
   2265,
   1000,
   3677,
   4000,
   2046,
   8398,
   2058,
   6745,
   2317,
   2160,
   10887,
   1012,
   102,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   

### Evaluate Functions for the Model

In [17]:
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy(predictions_and_labels: Tuple[np.array, np.array]):
  """
  Computes the accuracy of a model by comparing the predictions and labels.
  """
  predictions, labels = predictions_and_labels

  # Get highest prediction probability of each prediction if predictions are probabilities
  if len(predictions.shape) >= 2:
    predictions = np.argmax(predictions, axis=1)

  return accuracy_metric.compute(predictions=predictions, references=labels)

In [18]:
# Create example list of predictions and labels for testing the evaluate function
example_predictions_all_correct = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
example_predictions_one_wrong = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0])
example_labels = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

# Test the function
print(f"Accuracy when all predictions are correct: {compute_accuracy((example_predictions_all_correct, example_labels))}")
print(f"Accuracy when one prediction is wrong: {compute_accuracy((example_predictions_one_wrong, example_labels))}")

Accuracy when all predictions are correct: {'accuracy': 1.0}
Accuracy when one prediction is wrong: {'accuracy': 0.9}


### Model Training

In [ ]:
# define model and load the pretained model for sequence classification
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path= "distilbert/distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [22]:
# count the parameters in the model
def count_params(model):
    """
    Count the parameters of a PyTorch model.
    """
    trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_parameters = sum(p.numel() for p in model.parameters())

    return {"trainable_parameters": trainable_parameters, "total_parameters": total_parameters}

# Count the parameters of the model
count_params(model)

{'trainable_parameters': 66985770, 'total_parameters': 66985770}

In [24]:
# Create model output directory
from pathlib import Path

# Create models directory
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

# Create model save name
model_save_name = "distilbert-base-uncased-News-Category-Classifier"

# Create model save path
model_save_dir = Path(models_dir, model_save_name)

model_save_dir


# define the saved model path (huggingface model hub path)
# model_save_name = "distilbert-base-uncased-News-Category-Classifier"
# model_save_path = f"{huggingface_hub.whoami()['name']}/{model_save_name}"
# model_save_path

PosixPath('models/distilbert-base-uncased-News-Category-Classifier')

In [ ]:
# define training arguments for the model training
from transformers import TrainingArguments

print(f"[INFO] Saving model checkpoints to: {model_save_dir}")

training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=0.0001,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy="epoch", # was previously "evaluation_strategy"
    save_strategy="epoch",
    save_total_limit=3, # limit the total amount of save checkpoints (so we don't save num_epochs checkpoints)
    use_cpu=False, # set to False by default, will use CUDA GPU or MPS device if available
    seed=42, # set to 42 by default for reproducibility
    load_best_model_at_end=True, # load the best model when finished training
    logging_strategy="epoch", # log training results every epoch
    report_to="none", # optional: log experiments to Weights & Biases/other similar experimenting tracking services (we'll turn this off for now) 
    # push_to_hub=True # optional: automatically upload the model to the Hub (we'll do this manually later on)
    # hub_token="your_token_here" # optional: add your Hugging Face Hub token to push to the Hub (will default to huggingface-cli login)
    hub_private_repo=False # optional: make the uploaded model private (defaults to False)
)

[INFO] Saving model checkpoints to: models/distilbert-base-uncased-News-Category-Classifier
